In [1]:
import os
DATA = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if any(f.endswith(".h5") for f in files) and any(f.endswith("manifest.csv") for f in files):
        DATA = root
        break
assert DATA, "no folder with both the .h5 and the manifest: send Claude the find output"
print("data folder:", DATA, os.listdir(DATA))
%cd /kaggle/working
!rm -rf calib-cl && git clone -q https://github.com/safia-ahmad/calib-cl.git
%cd /kaggle/working/calib-cl
!pip install -q pytest
!python -m pytest -q

data folder: /kaggle/input/datasets/safiaahmadkhan/pcam-compact ['pcam_compact.h5', 'pcam_compact_manifest.csv']
/kaggle/working
/kaggle/working/calib-cl
................................                                         [100%]
=============================== warnings summary ===============================
tests/test_validate.py::test_known_answers_on_a_calibrated_model
tests/test_validate.py::test_underconfident_start_is_centred_first
  /usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
    opt_res = optimize.minimize(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
32 passed, 2 warnings in 33.24s


In [2]:
!python scripts/train_source.py --setting pcam_fwd --seed 0 --data {DATA} --out /kaggle/working/test_runs --epochs 1 --max_train 3000 --workers 4

device: cuda | run folder: /kaggle/working/test_runs/pcam_fwd_resnet18_seed0
slide_type       normal  tumour  All
domain role                         
source stream        20      14   34
       test          20      14   34
       train         45      32   77
       val_buf        5       3    8
       val_es         5       4    9
       val_temp       5       3    8
target dev            6       4   10
       stream        35      24   59
       test          18      12   30
All                 159     110  269
patches: train 3000, val_es 4285, val_temp 3781, S 1840, G 1720
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 192MB/s]
epoch 1: train loss 0.3868 | val NLL 0.5085 | val acc 0.757 | 4s
temperature T0 = 1.656

=== results (temperature-scaled) ===
S_source_test  acc 0.814  AUROC 0.903  Brier 0.1260  MCB -0.0107  DSC 0.111

In [3]:
!python scripts/build_streams.py --data {DATA} --check

Supply = patches available in the stream slides (src = source hospital, tgt = target).
Each design needs: gradual stream about 5n source + 5n target patches; no-drift twin 10n source.

 setting  seed  window  tumour_share  src_normal  src_tumour  tgt_normal  tgt_tumour                     status
pcam_fwd     0    1000           0.5       11169        5059       19069        9065                         OK
pcam_fwd     0    1000           0.3       11169        5059       19069        9065                         OK
pcam_fwd     0     750           0.5       11169        5059       19069        9065                         OK
pcam_fwd     0     750           0.3       11169        5059       19069        9065                         OK
pcam_fwd     0     500           0.5       11169        5059       19069        9065                         OK
pcam_fwd     0     500           0.3       11169        5059       19069        9065                         OK
pcam_fwd     1    1000         

In [4]:
!python scripts/build_streams.py --data {DATA} --out /kaggle/working/streams --window 500 --tumour_share 0.5

pcam_fwd seed 0: saved /kaggle/working/streams/pcam_fwd_seed0.npz | target share per window [0.0, 0.06, 0.2, 0.32, 0.39, 0.57, 0.67, 0.8, 0.93, 1.0]
pcam_fwd seed 1: saved /kaggle/working/streams/pcam_fwd_seed1.npz | target share per window [0.0, 0.05, 0.18, 0.33, 0.5, 0.55, 0.63, 0.81, 0.92, 1.0]
pcam_fwd seed 2: saved /kaggle/working/streams/pcam_fwd_seed2.npz | target share per window [0.0, 0.07, 0.22, 0.31, 0.48, 0.56, 0.68, 0.8, 0.93, 1.0]
pcam_fwd seed 3: saved /kaggle/working/streams/pcam_fwd_seed3.npz | target share per window [0.0, 0.08, 0.2, 0.31, 0.46, 0.54, 0.66, 0.8, 0.94, 1.0]
pcam_fwd seed 4: saved /kaggle/working/streams/pcam_fwd_seed4.npz | target share per window [0.0, 0.05, 0.19, 0.32, 0.39, 0.5, 0.7, 0.81, 0.93, 1.0]
pcam_fwd seed 5: saved /kaggle/working/streams/pcam_fwd_seed5.npz | target share per window [0.0, 0.07, 0.23, 0.32, 0.44, 0.51, 0.68, 0.82, 0.95, 1.0]
pcam_rev seed 0: saved /kaggle/working/streams/pcam_rev_seed0.npz | target share per window [0.0, 0.07

In [5]:
OUT = "/kaggle/working/runs"
for setting in ["pcam_fwd", "pcam_rev"]:
    for seed in range(6):
        print(f"\n########## {setting} seed {seed} ##########")
        !python scripts/train_source.py --setting {setting} --seed {seed} --model resnet18 --data {DATA} --out {OUT} --workers 4


########## pcam_fwd seed 0 ##########
device: cuda | run folder: /kaggle/working/runs/pcam_fwd_resnet18_seed0
slide_type       normal  tumour  All
domain role                         
source stream        20      14   34
       test          20      14   34
       train         45      32   77
       val_buf        5       3    8
       val_es         5       4    9
       val_temp       5       3    8
target dev            6       4   10
       stream        35      24   59
       test          18      12   30
All                 159     110  269
patches: train 37070, val_es 4285, val_temp 3781, S 1840, G 1720
epoch 1: train loss 0.2244 | val NLL 0.3273 | val acc 0.855 | 19s
epoch 2: train loss 0.1513 | val NLL 0.2627 | val acc 0.884 | 18s
epoch 3: train loss 0.1284 | val NLL 0.1815 | val acc 0.921 | 18s
epoch 4: train loss 0.1059 | val NLL 0.2228 | val acc 0.911 | 18s
epoch 5: train loss 0.0971 | val NLL 0.2353 | val acc 0.907 | 18s
epoch 6: train loss 0.0852 | val NLL 0.1862 | val 